In [2]:
import os
from typing import List
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

In [3]:
# ── STEP 1: Define Output Structure ──────────────────────────────
# WHY: We want structured JSON back, not raw text.
# Pydantic ensures the model's output is typed and validated.

class RiskClause(BaseModel):
    clause:         str = Field(description='The risky clause found')
    risk_level:     str = Field(description='HIGH / MEDIUM / LOW')
    reason:         str = Field(description='Why this clause is risky')
    recommendation: str = Field(description='What to do about it')

class ContractAnalysis(BaseModel):
    summary:      str             = Field(description='One sentence summary')
    overall_risk: str             = Field(description='HIGH / MEDIUM / LOW')
    risks:        List[RiskClause] = Field(description='List of risky clauses')
    safe_to_sign: bool            = Field(description='True if safe to sign as-is')

In [4]:
# ── STEP 2: Few-Shot Example ─────────────────────────────────────
# WHY: Without an example, the model may hallucinate format.
# One worked example anchors it to our exact output structure.

examples = [{
    'contract': 'Vendor may terminate with 7 days notice. Client liable for all indirect damages.',
    'output': '''{"summary": "Vendor agreement with unfavorable termination and liability terms.",
  "overall_risk": "HIGH",
  "risks": [
    {"clause": "Vendor may terminate with 7 days notice",
     "risk_level": "HIGH",
     "reason": "7 days is too short. No business continuity possible.",
     "recommendation": "Negotiate minimum 90 days notice."},
    {"clause": "Client liable for all indirect damages",
     "risk_level": "HIGH",
     "reason": "Unlimited liability exposure for client.",
     "recommendation": "Cap liability to contract value only."}
  ],
  "safe_to_sign": false}'''
}]


In [5]:
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{contract}'),
    ('ai',    '{output}')
])

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)


In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
# ── STEP 3: Build the Chain ──────────────────────────────────────
# WHY: This is the full LCEL pipeline.
# prompt → formats messages → model → generates response → parser → structured JSON

parser = JsonOutputParser(pydantic_object=ContractAnalysis)

prompt = ChatPromptTemplate.from_messages([
    ('system', '''You are a senior legal risk analyst at EY.
Analyze the contract text and return ONLY valid JSON matching this schema:
{format_instructions}'''),
    few_shot,                          # ← few-shot example injected here
    ('human', '{contract_text}')       # ← actual user input
]).partial(format_instructions=parser.get_format_instructions())



model = ChatOpenAI(model='gpt-4o', temperature=0)

# THE CHAIN — 3 components, 1 line
chain = prompt | model | parser

In [14]:
# ── STEP 4: Run It ───────────────────────────────────────────────

contract = """
SOFTWARE SERVICES AGREEMENT

1. Payment: Client shall pay within Net 90 days of invoice.
2. IP Ownership: All work products created remain property of the Vendor.
3. Termination: Either party may terminate with 7 days written notice.
4. Liability: Vendor's liability is capped at INR 10,000 regardless of damages.
5. Governing Law: Disputes resolved in Vendor's home jurisdiction.
6. Data: Vendor may share client data with third-party sub-processors without notice.
"""

result = chain.invoke({'contract_text': contract})
result


{'summary': 'Software Services Agreement with several high-risk clauses for the client.',
 'overall_risk': 'HIGH',
 'risks': [{'clause': 'IP Ownership: All work products created remain property of the Vendor.',
   'risk_level': 'HIGH',
   'reason': 'Client does not retain ownership of work products, limiting their use and control.',
   'recommendation': 'Negotiate joint ownership or a license to use the work products.'},
  {'clause': 'Termination: Either party may terminate with 7 days written notice.',
   'risk_level': 'HIGH',
   'reason': '7 days notice is insufficient for business continuity planning.',
   'recommendation': 'Negotiate a longer notice period, such as 30 or 60 days.'},
  {'clause': "Liability: Vendor's liability is capped at INR 10,000 regardless of damages.",
   'risk_level': 'HIGH',
   'reason': 'The liability cap is too low and may not cover potential damages.',
   'recommendation': 'Negotiate a higher liability cap that reflects the potential risk exposure.'},
  {

In [15]:
contracts = [
    {"contract_text": """
    NDA — NON DISCLOSURE AGREEMENT
    1. Confidentiality: Both parties agree to keep all shared information confidential for 1 year.
    2. Exclusions: Information already in public domain is not covered.
    3. Breach Penalty: INR 50,000 flat penalty for any breach.
    4. Governing Law: Courts of Mumbai, India.
    5. Termination: Agreement ends automatically after 1 year with no renewal clause.
    """},

    {"contract_text": """
    FREELANCE DEVELOPER CONTRACT
    1. Payment: Client pays INR 500 per hour. Invoices due Net 15 days.
    2. IP Ownership: All code written by freelancer remains freelancer's property until full payment received.
    3. Revisions: Client entitled to unlimited revisions with no time limit.
    4. Termination: Client may terminate any time. No kill fee or compensation for work in progress.
    5. Liability: Freelancer liable for all bugs found within 5 years of delivery.
    6. Non-compete: Freelancer cannot work with any client in same industry for 3 years.
    """},

    {"contract_text": """
    CLOUD SERVICES AGREEMENT — SAAS VENDOR
    1. Uptime SLA: Vendor guarantees 99.9% uptime. Downtime credit capped at 1 month subscription fee.
    2. Data Ownership: Customer owns all data uploaded to the platform.
    3. Termination: Either party may terminate with 30 days notice.
    4. Price Changes: Vendor may change pricing with 7 days notice.
    5. Auto Renewal: Contract auto-renews annually unless cancelled 60 days before renewal date.
    6. Data Deletion: Upon termination, customer data deleted within 90 days.
    7. Liability Cap: Total liability capped at 3 months of fees paid.
    """},

    {"contract_text": """
    EMPLOYMENT CONTRACT — SENIOR ANALYST
    1. Salary: INR 12 LPA fixed. No variable component.
    2. Notice Period: 90 days from either side.
    3. Non-Compete: Employee cannot join any competitor for 2 years post resignation.
    4. IP: All work created during employment, including personal side projects, belongs to the company.
    5. Probation: 6 months probation. Company may terminate during probation with zero notice.
    6. Overtime: No additional compensation for overtime or weekend work.
    7. Jurisdiction: All disputes resolved in employer's city of registration.
    """},

    {"contract_text": """
    VENDOR PROCUREMENT AGREEMENT
    1. Payment Terms: Net 60 days from invoice date.
    2. Delivery: Vendor must deliver within agreed timelines or face 2% penalty per week of delay.
    3. Quality: Rejected goods must be replaced within 7 days at vendor's cost.
    4. Termination: Buyer may terminate with 14 days notice for any reason. No compensation to vendor.
    5. Force Majeure: Delays due to natural disasters or government orders are excused for both parties.
    6. Governing Law: Indian Contract Act 1872. Disputes via arbitration in Delhi.
    7. Warranty: 1 year warranty on all delivered goods.
    """}
]

results = chain.batch(contracts)

In [16]:
results

[{'summary': 'NDA with limited confidentiality duration and flat breach penalty.',
  'overall_risk': 'MEDIUM',
  'risks': [{'clause': 'Confidentiality: Both parties agree to keep all shared information confidential for 1 year.',
    'risk_level': 'MEDIUM',
    'reason': 'Confidentiality duration may be too short for sensitive information.',
    'recommendation': 'Consider extending confidentiality period to 3-5 years.'},
   {'clause': 'Breach Penalty: INR 50,000 flat penalty for any breach.',
    'risk_level': 'MEDIUM',
    'reason': 'Flat penalty may not cover actual damages from a breach.',
    'recommendation': 'Include a clause for actual damages or higher penalty for significant breaches.'},
   {'clause': 'Termination: Agreement ends automatically after 1 year with no renewal clause.',
    'risk_level': 'LOW',
    'reason': 'No option for renewal may limit ongoing protection.',
    'recommendation': 'Add a renewal clause to extend the agreement if necessary.'}],
  'safe_to_sign': 

In [17]:
import pandas as pd
rows = []
for i, result in enumerate(results):
    for risk in result['risks']:
        rows.append({
            'contract_no':    i + 1,
            'summary':        result['summary'],
            'overall_risk':   result['overall_risk'],
            'safe_to_sign':   result['safe_to_sign'],
            'clause':         risk['clause'],
            'risk_level':     risk['risk_level'],
            'reason':         risk['reason'],
            'recommendation': risk['recommendation']
        })

df_exploded = pd.DataFrame(rows)
df_exploded.head(20) 

,contract_no,summary,overall_risk,safe_to_sign,clause,risk_level,reason,recommendation
0,1,NDA with limited confidentiality duration and ...,MEDIUM,True,Confidentiality: Both parties agree to keep al...,MEDIUM,Confidentiality duration may be too short for ...,Consider extending confidentiality period to 3...
1,1,NDA with limited confidentiality duration and ...,MEDIUM,True,"Breach Penalty: INR 50,000 flat penalty for an...",MEDIUM,Flat penalty may not cover actual damages from...,Include a clause for actual damages or higher ...
2,1,NDA with limited confidentiality duration and ...,MEDIUM,True,Termination: Agreement ends automatically afte...,LOW,No option for renewal may limit ongoing protec...,Add a renewal clause to extend the agreement i...
3,2,Freelance developer contract with several high...,HIGH,False,IP Ownership: All code written by freelancer r...,MEDIUM,Potential delay in IP transfer could affect cl...,Specify a clear timeline for payment and IP tr...
4,2,Freelance developer contract with several high...,HIGH,False,Revisions: Client entitled to unlimited revisi...,HIGH,Unlimited revisions can lead to excessive unpa...,Limit revisions to a reasonable number or time...
5,2,Freelance developer contract with several high...,HIGH,False,Termination: Client may terminate any time. No...,HIGH,Freelancer may not be compensated for work alr...,Include a kill fee or compensation clause for ...
6,2,Freelance developer contract with several high...,HIGH,False,Liability: Freelancer liable for all bugs foun...,HIGH,Extensive liability period increases risk for ...,"Limit liability period to a shorter duration, ..."
7,2,Freelance developer contract with several high...,HIGH,False,Non-compete: Freelancer cannot work with any c...,HIGH,Restricts freelancer's ability to find future ...,Negotiate a shorter non-compete period or remo...
8,3,Cloud services agreement with potential risks ...,MEDIUM,False,Price Changes: Vendor may change pricing with ...,MEDIUM,Short notice period for price changes can lead...,Negotiate a longer notice period for price cha...
9,3,Cloud services agreement with potential risks ...,MEDIUM,False,Liability Cap: Total liability capped at 3 mon...,MEDIUM,Liability cap may not cover potential losses f...,Consider negotiating a higher liability cap or...


## Use case 2 : Resume Scanner

In [18]:
# ── STEP 1: Define Output Structure ──────────────────────────────
# WHY: HR needs a consistent scorecard, not a paragraph of text.
# Pydantic enforces the exact fields every time — no hallucinated fields.

class ResumeScore(BaseModel):
    candidate_name:    str       = Field(description='Name extracted from resume')
    match_score:       int       = Field(description='Overall match score 0-100')
    matched_skills:    List[str] = Field(description='Skills present in both JD and resume')
    missing_skills:    List[str] = Field(description='Skills required in JD but missing in resume')
    experience_fit:    str       = Field(description='STRONG / PARTIAL / WEAK')
    recommendation:    str       = Field(description='SHORTLIST / MAYBE / REJECT')
    one_line_summary:  str       = Field(description='One sentence hiring manager summary')

In [19]:
# ── STEP 2: Few-Shot Calibration Example ─────────────────────────
# WHY: Without calibration, model scores are inconsistent.
# One scored example teaches the model what a 75 vs 40 vs 90 looks like.

examples = [{
    'jd': 'Role: Python Developer. Required: Python, FastAPI, PostgreSQL, Docker, 3+ years experience.',
    'resume': 'John Doe. 4 years Python. Built REST APIs with Flask. Used MySQL. No Docker experience.',
    'output': '{"candidate_name": "John Doe", "match_score": 65, "matched_skills": ["Python", "REST APIs", "SQL databases"], "missing_skills": ["FastAPI", "PostgreSQL", "Docker"], "experience_fit": "PARTIAL", "recommendation": "MAYBE", "one_line_summary": "Strong Python background but missing key infrastructure skills; worth a call if pipeline is thin."}'
}]


example_prompt = ChatPromptTemplate.from_messages([
    ('human', 'JD: {jd}\n\nResume: {resume}'),
    ('ai',    '{output}')
])


few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

In [20]:
parser = JsonOutputParser(pydantic_object=ResumeScore)

prompt = ChatPromptTemplate.from_messages([
    ('system', '''You are a senior technical recruiter at EY .
Evaluate the resume against the job description and return ONLY valid JSON.
Be strict on technical skills. Be fair on experience years (+/- 1 year is acceptable).
Schema: {format_instructions}'''),
    few_shot,
    ('human', 'JD:\n{jd}\n\nResume:\n{resume}')
]).partial(format_instructions=parser.get_format_instructions())

model = ChatOpenAI(model='gpt-4o', temperature=0)

screen_chain = prompt | model | parser

In [21]:
 #── STEP 4: Define a Real JD ──────────────────────────────────────

JD = """
Role: AI Engineer — EY =, Mumbai

Required Skills:
- Python (4+ years)
- Machine Learning (scikit-learn, XGBoost)
- Deep Learning (PyTorch or TensorFlow)
- LangChain / LLM application development
- REST API development (FastAPI or Flask)
- MLflow or any MLOps tool
- SQL and data handling (Pandas)

Good to have: Azure ML, Docker, NLP experience
Experience: 3-6 years
"""

screen_chain.invoke({
    'jd': JD,
    'resume': 'John Doe. 4 years Python. Built REST APIs with Flask. Used MySQL. No Docker experience.'
})

{'candidate_name': 'John Doe',
 'match_score': 50,
 'matched_skills': ['Python', 'REST API development', 'Flask', 'SQL'],
 'missing_skills': ['Machine Learning',
  'Deep Learning',
  'LangChain',
  'LLM application development',
  'MLflow',
  'MLOps',
  'Pandas',
  'Azure ML',
  'Docker',
  'NLP'],
 'experience_fit': 'WEAK',
 'recommendation': 'REJECT',
 'one_line_summary': 'Lacks essential AI and MLOps skills required for the AI Engineer role.'}

In [22]:
resumes = [
    {
        'jd': JD,
        'resume': '''
        Rahul Sharma | Mumbai
        7 years experience. Python expert.
        Built ML pipelines with scikit-learn and XGBoost.
        Deep learning with PyTorch. NLP with HuggingFace transformers.
        Built LangChain-based RAG systems. FastAPI REST APIs.
        MLflow for experiment tracking. Pandas, SQL daily use.
        Azure ML certified.
        '''
    },
    {
        'jd': JD,
        'resume': '''
        Priya Patel | Bangalore
        2 years experience. Python and data analysis.
        Used scikit-learn for classification tasks.
        Basic Flask APIs. Familiar with Pandas and SQL.
        No deep learning or LLM experience.
        '''
    },
    {
        'jd': JD,
        'resume': '''
        Amit Kumar | Delhi
        5 years experience. Python and ML.
        TensorFlow for image classification projects.
        FastAPI, Pandas, SQL. Docker experience.
        Started exploring LangChain recently.
        No MLflow but used custom experiment tracking.
        '''
    }
]

# All 3 screened in parallel
results = screen_chain.batch(resumes)

In [23]:
# ── STEP 6: Display Scorecard ─────────────────────────────────────

for i, r in enumerate(results, 1):
    bar = '█' * (r['match_score'] // 10) + '░' * (10 - r['match_score'] // 10)
    rec_emoji = {'SHORTLIST': '✅', 'MAYBE': '🟡', 'REJECT': '❌'}.get(r['recommendation'], '❓')

    print(f"{'='*60}")
    print(f"  Candidate  : {r['candidate_name']}")
    print(f"  Score      : [{bar}] {r['match_score']}/100")
    print(f"  Exp Fit    : {r['experience_fit']}")
    print(f"  Decision   : {rec_emoji} {r['recommendation']}")
    print(f"  Matched    : {', '.join(r['matched_skills'])}")
    print(f"  Missing    : {', '.join(r['missing_skills'])}")
    print(f"  Summary    : {r['one_line_summary']}")
    print()

  Candidate  : Rahul Sharma
  Score      : [█████████░] 95/100
  Exp Fit    : STRONG
  Decision   : ✅ SHORTLIST
  Matched    : Python, Machine Learning, scikit-learn, XGBoost, Deep Learning, PyTorch, LangChain, REST API development, FastAPI, MLflow, SQL, Pandas, Azure ML, NLP
  Missing    : TensorFlow, Docker
  Summary    : Highly experienced AI Engineer with strong skills in Python, ML, and deep learning; excellent fit for the role.

  Candidate  : Priya Patel
  Score      : [████░░░░░░] 40/100
  Exp Fit    : WEAK
  Decision   : ❌ REJECT
  Matched    : Python, scikit-learn, Flask, Pandas, SQL
  Missing    : XGBoost, Deep Learning, LangChain, FastAPI, MLflow, MLOps, Azure ML, Docker, NLP
  Summary    : Insufficient experience and missing critical AI and MLOps skills for the AI Engineer role.

  Candidate  : Amit Kumar
  Score      : [████████░░] 85/100
  Exp Fit    : STRONG
  Decision   : ✅ SHORTLIST
  Matched    : Python, Machine Learning, Deep Learning, TensorFlow, FastAPI, SQL, Pand

In [24]:
# ── BONUS: Sort candidates by score ──────────────────────────────
# WHY: Once you have structured JSON output, you can sort,
# filter, export to Excel — treat it like any Python data.

ranked = sorted(results, key=lambda x: x['match_score'], reverse=True)

print('🏆 RANKED CANDIDATES')
print('─' * 40)
for rank, r in enumerate(ranked, 1):
    print(f"{rank}. {r['candidate_name']:20s} Score: {r['match_score']:3d}  →  {r['recommendation']}")

🏆 RANKED CANDIDATES
────────────────────────────────────────
1. Rahul Sharma         Score:  95  →  SHORTLIST
2. Amit Kumar           Score:  85  →  SHORTLIST
3. Priya Patel          Score:  40  →  REJECT
